# Vector Context Provider

This notebook uses `Neo4jContextProvider` from the [`agent-framework-neo4j`](https://github.com/neo4j-labs/neo4j-maf-provider) package — a MAF context provider that connects your agent to a Neo4j knowledge graph. Before each LLM invocation, the provider automatically searches Neo4j and injects the results into the agent's context.

Here you'll configure the provider with **vector search** (semantic similarity). Vector search understands meaning — it can find conceptually related content even when exact keywords don't match. The provider embeds the user's query, searches the Neo4j vector index for semantically similar chunks, and injects the top results.

***

Load the environment variables and import the required modules.

In [1]:
import sys
sys.path.insert(0, '../shared')

from azure.identity import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential

from agent_framework.azure import AzureAIClient
from agent_framework_neo4j import (
    AzureAIEmbedder,
    AzureAISettings,
    Neo4jContextProvider,
    Neo4jSettings,
)

from config import get_agent_config

Get the configuration and create the embedder for vector search.

In [2]:
# Get configuration from environment
config = get_agent_config()
neo4j_settings = Neo4jSettings()
azure_settings = AzureAISettings()

# Create embedder for converting queries to vectors
sync_credential = DefaultAzureCredential()

embedder = AzureAIEmbedder(
    endpoint=azure_settings.inference_endpoint,
    credential=sync_credential,
    model=azure_settings.embedding_model,
)

print(f"Neo4j URI: {neo4j_settings.uri}")
print(f"Vector Index: {neo4j_settings.vector_index_name}")
print(f"Embedding Model: {azure_settings.embedding_model}")

Neo4j URI: neo4j+s://72505174.databases.neo4j.io
Vector Index: chunkEmbeddings
Embedding Model: text-embedding-3-small


## Create the Vector Context Provider

Create a `Neo4jContextProvider` with `index_type="vector"`. The provider will:
1. Embed the user's query using the Azure AI embedder
2. Search the Neo4j vector index for semantically similar chunks
3. Inject the results into the agent's context

In [3]:
# Create context provider with vector search
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.vector_index_name,
    index_type="vector",
    embedder=embedder,
    top_k=5,
    context_prompt=(
        "## Semantic Search Results\n"
        "Use the following semantically relevant information from the "
        "knowledge graph to answer questions:"
    ),
)

> Vector search requires an `embedder` to convert queries into vectors. The `index_type="vector"` tells the provider to use the `VectorRetriever` from neo4j-graphrag.

***

## Run the Agent with Vector Context

Create an agent that receives automatically injected semantic search results. The agent is created once and reused for all queries below.

In [4]:
# Connect provider and create agent
credential = AzureCliCredential()
await provider.__aenter__()
print("Connected to Neo4j with vector search!\n")

client = AzureAIClient(
    project_endpoint=config.project_endpoint,
    model_deployment_name=config.model_name,
    credential=credential,
)

agent = client.as_agent(
    name="workshop-vector-agent",
    instructions=(
        "You are a helpful assistant that answers questions about companies "
        "using the provided semantic search context. Be concise and accurate. "
        "When the context contains relevant information, cite it in your response."
    ),
    context_providers=[provider],
)

session = agent.create_session()

async def ask(query):
    """Run a query against the agent and print the response."""
    print(f"User: {query}\n")
    print("Assistant: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

print(f"Agent '{agent.name}' created")

Connected to Neo4j with vector search!

Agent 'workshop-vector-agent' created


In [5]:
await ask("What are the main business activities of tech companies?")

User: What are the main business activities of tech companies?

Assistant: Tech companies primarily engage in the following business activities:

1. **Software Development**: Creating applications, operating systems, and enterprise software solutions for various platforms.
  
2. **Hardware Production**: Designing and manufacturing computing devices like computers, smartphones, and peripherals.

3. **Cloud Computing**: Offering cloud services for data storage, processing, and software applications, enabling scalable infrastructure for businesses.

4. **E-commerce**: Facilitating online transactions, retail, and marketplaces through web platforms.

5. **Digital Services**: Providing services such as social media, online advertising, and content streaming.

6. **Data Analytics**: Analyzing large datasets to derive insights for businesses and consumers, often using machine learning and AI technologies.

7. **Cybersecurity**: Developing solutions to protect data and networks from cyber thre

> Vector search found semantically similar content even without exact keyword matches. Try queries that describe concepts rather than specific terms.

***

Run the cells below to experiment with different queries. You can also modify the queries or add new cells.

***

[View the complete code](../financial_data_load/solution_srcs/06_02_vector_context_provider.py)

[Move on to the Graph-Enriched Provider Notebook](02_graph_enriched_provider.ipynb)

In [6]:
await ask("Describe challenges and risks in the technology sector")

User: Describe challenges and risks in the technology sector

Assistant: The technology sector faces several challenges and risks, including:

1. **Cybersecurity Threats**: The increasing sophistication of cyberattacks can lead to data breaches, financial losses, and reputational damage. Companies must continuously invest in security measures to protect sensitive information.

2. **Data Privacy Regulations**: Compliance with laws such as GDPR and CCPA is complex and costly. Missteps can result in significant fines and legal liabilities.

3. **Evolving Technology Landscape**: Rapid technological advancements require companies to continuously innovate or risk obsolescence. Keeping up with trends such as AI, IoT, and cloud computing can be challenging.

4. **Supply Chain Vulnerabilities**: Disruptions in the supply chain, such as those caused by geopolitical tensions or natural disasters, can impact production and delivery.

5. **Competition**: The sector is highly competitive, with new e

In [7]:
await ask("How do companies generate revenue and measure performance?")

User: How do companies generate revenue and measure performance?

Assistant: Companies generate revenue and measure performance through various methods:

### Revenue Generation:

1. **Sales of Goods and Services**: 
   - **Retail Sales**: Companies sell products directly to consumers or businesses.
   - **Subscription Services**: Revenue is generated through recurring fees for access to services (e.g., SaaS companies).

2. **Advertising Revenue**: 
   - Platforms may generate income by displaying ads to users, charging advertisers based on impressions or clicks.

3. **Licensing Fees**: 
   - Companies may license their intellectual property, such as software or patents, to others for a fee.

4. **Transaction Fees**: 
   - Payment processors and marketplace platforms collect fees for every transaction facilitated.

5. **Investment Income**: 
   - Companies may earn income from interests, dividends, or capital gains from investments.

6. **Partnerships and Sponsorships**: 
   - Revenue c

In [8]:
# Cleanup
await provider.__aexit__(None, None, None)
await credential.close()
embedder.close()